In [1]:
import re
import pandas as pd
import torch
import torch.nn as nn

from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader


In [2]:
df = pd.read_csv("dataset-exemplos.csv", sep=";")

# Show first rows
print(df.head())

# Keep only useful columns
df = df[["Text", "Label"]].copy()

# Remove missing values
df = df.dropna(subset=["Text", "Label"])

# Make sure text is string
df["Text"] = df["Text"].astype(str)

print("\nDataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

     ID                                               Text   Label
0  D1-1  It is an approximation useful in chemistry, bu...   Human
1  D1-2  PET scanning, or Positron Emission Tomography,...    Meta
2  D1-3  Positron Emission Tomography (PET) scanning is...  Google
3  D1-4  Thermonuclear fusion is the process of combini...    Meta
4  D1-5  These nutrients are needed to keep bones, teet...   Human

Dataset shape: (125, 2)

Columns: ['Text', 'Label']


In [3]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Apply text cleaning
df["Text"] = df["Text"].apply(clean_text)

In [4]:
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["Label"])

print("\nClasses:")
for i, class_name in enumerate(encoder.classes_):
    print(f"{i} -> {class_name}")



Classes:
0 -> Anthropic
1 -> Google
2 -> Human
3 -> Meta
4 -> OpenAI


In [5]:
texts = df["Text"].tolist()
labels = df["label"].values

In [6]:
def build_vocab(texts, max_words=8000):
    counter = Counter()

    # Count word frequencies
    for text in texts:
        counter.update(text.split())

    # Keep the most frequent words
    most_common = counter.most_common(max_words)

    # Reserve:
    # 0 -> <PAD>
    # 1 -> <UNK>
    word_index = {"<PAD>": 0, "<UNK>": 1}

    for i, (word, _) in enumerate(most_common, start=2):
        word_index[word] = i

    return word_index

word_index = build_vocab(texts, max_words=8000)

In [7]:
def encode_text(text, word_index, max_len=120):
    tokens = text.split()

    # Convert each token into its index
    sequence = [word_index.get(word, 1) for word in tokens]  # 1 = <UNK>

    # Truncate long sequences
    sequence = sequence[:max_len]

    # Pad short sequences
    if len(sequence) < max_len:
        sequence += [0] * (max_len - len(sequence))  # 0 = <PAD>

    return sequence

max_len = 120
X_seq = [encode_text(text, word_index, max_len=max_len) for text in texts]
y = labels

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X_seq,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
class TextDatasetLSTM(Dataset):
    def __init__(self, X, y):
        # X contains token indices, so dtype must be long
        self.X = torch.tensor(X, dtype=torch.long)

        # Labels for classification
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TextDatasetLSTM(X_train, y_train)
val_dataset = TextDatasetLSTM(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [10]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()

        # Embedding layer converts token ids into dense vectors
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        # Bidirectional LSTM reads the sequence in both directions
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        # Dropout reduces overfitting
        self.dropout = nn.Dropout(0.3)

        # Final classification layer
        # hidden_dim * 2 because the LSTM is bidirectional
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # Convert token ids to embeddings
        embedded = self.embedding(x)

        # Pass embeddings through BiLSTM
        _, (hidden, _) = self.lstm(embedded)

        # Take the last hidden state from forward and backward directions
        last_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)

        # Apply dropout
        last_hidden = self.dropout(last_hidden)

        # Compute class scores
        out = self.fc(last_hidden)

        return out

In [11]:
vocab_size = len(word_index)
embed_dim = 64
hidden_dim = 128
num_classes = len(encoder.classes_)

model = LSTMClassifier(vocab_size, embed_dim, hidden_dim, num_classes)




In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [13]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

Epoch 1/10 - Loss: 1.6083
Epoch 2/10 - Loss: 1.5003
Epoch 3/10 - Loss: 1.4482
Epoch 4/10 - Loss: 1.3543
Epoch 5/10 - Loss: 1.5117
Epoch 6/10 - Loss: 1.3807
Epoch 7/10 - Loss: 1.4668
Epoch 8/10 - Loss: 1.2920
Epoch 9/10 - Loss: 1.1969
Epoch 10/10 - Loss: 1.2270


In [14]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

accuracy = correct / total
print(f"\nValidation accuracy: {accuracy:.4f}")


Validation accuracy: 0.4400


In [15]:
model.eval()

with torch.no_grad():
    X_val_tensor = torch.tensor(X_val, dtype=torch.long)
    outputs = model(X_val_tensor)
    _, predicted = torch.max(outputs, 1)

predicted_labels = encoder.inverse_transform(predicted.numpy())
true_labels = encoder.inverse_transform(y_val)

results_df = pd.DataFrame({
    "True Label": true_labels,
    "Predicted Label": predicted_labels
})

print("\nSample predictions:")
print(results_df.head(10))


Sample predictions:
  True Label Predicted Label
0       Meta           Human
1      Human           Human
2      Human           Human
3      Human           Human
4     Google       Anthropic
5      Human           Human
6     Google           Human
7     OpenAI           Human
8  Anthropic       Anthropic
9      Human           Human
